# 17 · Identify gene and guide modules

Knockouts that move the same genes in the same direction are grouped into
**guide modules**; genes moved by the same knockouts are grouped into **gene
modules**. Both are Leiden clusterings of the effect-size matrix from notebook
16 — once over its rows, once over its columns.

**Reads** `par_selected_coef_matrix_file`.
**Writes** `par_guideModules_file` and `par_geneModules_file`.

The guide modules are the `K_0` … `K_5` that every downstream table is keyed
by. The number of modules is the result of the clustering, not a setting:
`par_guide_module_resolution` and `par_gene_module_resolution` are what you
control.

Leiden labels are not stable across runs or library versions. Re-running this
notebook can renumber the modules even when the partition is nearly unchanged,
which affects every table keyed by `K_0` … `K_5`.

## Setup

In [ ]:
from libraries import *
from parameters import *
from pathlib import Path

os.chdir(projectDir)

In [ ]:
coefs = pd.read_csv(par_selected_coef_matrix_file, index_col=0)
print(f"effect-size matrix: {coefs.shape[0]} knockouts x {coefs.shape[1]} genes")

## Guide modules

Each knockout is a point; its coordinates are its effects across genes.

In [ ]:
guides = sc.AnnData(X=coefs.to_numpy())
guides.obs_names = coefs.index

sc.pp.pca(guides, n_comps=min(50, min(guides.shape) - 1), svd_solver="arpack")
sc.pp.neighbors(guides)
sc.tl.leiden(guides, resolution=par_guide_module_resolution)
sc.tl.umap(guides)

guide_modules = pd.DataFrame({
    "GuideName": coefs.index,
    "GuideGroup": guides.obs["leiden"].to_numpy(),
})
n_guide_modules = guide_modules.GuideGroup.nunique()
print(f"guide modules at resolution {par_guide_module_resolution}: {n_guide_modules}")
print(guide_modules.GuideGroup.value_counts().sort_index().to_string())

sc.pl.umap(guides, color="leiden", size=100, legend_loc="on data")

## Gene modules

The same clustering over the transpose: each gene is a point, its coordinates
its response across knockouts.

In [ ]:
genes = sc.AnnData(X=coefs.transpose().to_numpy())
genes.obs_names = coefs.columns

sc.pp.pca(genes, n_comps=min(50, min(genes.shape) - 1), svd_solver="arpack")
sc.pp.neighbors(genes)
sc.tl.leiden(genes, resolution=par_gene_module_resolution)
sc.tl.umap(genes)

gene_modules = pd.DataFrame({
    "GeneName": coefs.columns,
    "GeneGroup": genes.obs["leiden"].to_numpy(),
})
n_gene_modules = gene_modules.GeneGroup.nunique()
print(f"gene modules at resolution {par_gene_module_resolution}: {n_gene_modules}")
print(gene_modules.GeneGroup.value_counts().sort_index().to_string())

sc.pl.umap(genes, color="leiden", size=50, legend_loc="on data")

## Write

Notebook 18 reads `GuideName` and `GuideGroup` from the guide table, and takes
the first column of the gene table as the response gene list.

In [ ]:
Path(par_guideModules_file).parent.mkdir(parents=True, exist_ok=True)

guide_modules.to_csv(par_guideModules_file)
print(f"written: {par_guideModules_file}  ({n_guide_modules} modules)")

gene_modules.to_csv(par_geneModules_file)
print(f"written: {par_geneModules_file}  ({n_gene_modules} modules)")